In [0]:

# ============================================================
# CELDA 1. PREPARACIÓN DE VARIABLES OPERATIVAS GOLD
# ============================================================

# Preparamos los vuelos para nuestros modelos XGBoost.
# Utilizaremos información programada del vuelo y,
# posteriormente, el pronóstico del clima del día anterior.
#
# No utilizaremos el retraso real como predictor.

from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

# Tablas de origen y destino
TABLA_VUELOS = "workspace.silver_flights.flights_clean"
TABLA_AEROPUERTOS = "workspace.bronze_flights.airport_coordinates"
TABLA_GOLD = "workspace.gold_flights.flights_features"

# ------------------------------------------------------------
# 1. CARGAR LOS VUELOS LIMPIOS
# ------------------------------------------------------------

vuelos = spark.table(TABLA_VUELOS)

# Consultamos las zonas horarias de los aeropuertos.
aeropuertos = spark.table(TABLA_AEROPUERTOS)

# Mostramos las columnas para identificar correctamente
# los nombres de los campos disponibles.
print("Columnas de aeropuertos:")
print(aeropuertos.columns)

# La tabla de coordenadas debe contener:
# airport y timezone
zonas = (
    aeropuertos
    .select(
        F.col("airport").alias("origin"),
        F.col("timezone").alias("origin_timezone")
    )
    .dropDuplicates(["origin"])
)

# Asociamos cada vuelo con la zona horaria de su origen.
df_gold = vuelos.join(
    F.broadcast(zonas),
    on="origin",
    how="left"
)

sin_zona = df_gold.filter(
    F.col("origin_timezone").isNull()
).count()

print("Vuelos sin zona horaria:", sin_zona)

if sin_zona > 0:
    raise ValueError(
        "Hay aeropuertos sin zona horaria. Revisar antes de continuar."
    )

# ------------------------------------------------------------
# 2. CONSTRUIR LA FECHA Y HORA LOCAL DE SALIDA
# ------------------------------------------------------------

# Ejemplo: 930 significa 09:30.
# Utilizamos la fecha y hora programadas del vuelo.

df_gold = df_gold.withColumn(
    "scheduled_departure_local",
    F.to_timestamp(
        F.concat(
            F.date_format(
                F.col("flight_date"),
                "yyyy-MM-dd"
            ),
            F.lit(" "),
            F.lpad(
                F.col("departure_hour").cast("string"),
                2,
                "0"
            ),
            F.lit(":"),
            F.lpad(
                F.col("departure_minute").cast("string"),
                2,
                "0"
            ),
            F.lit(":00")
        ),
        "yyyy-MM-dd HH:mm:ss"
    )
)

# ------------------------------------------------------------
# 3. CONVERTIR LA SALIDA PROGRAMADA A UTC
# ------------------------------------------------------------

# Los horarios BTS son locales, mientras que nuestros
# pronósticos meteorológicos están guardados en UTC.
# Esta conversión permite relacionar ambos correctamente.

df_gold = df_gold.withColumn(
    "scheduled_departure_utc",
    F.to_utc_timestamp(
        F.col("scheduled_departure_local"),
        F.col("origin_timezone")
    )
)

# ------------------------------------------------------------
# 4. DEFINIR LA HORA DEL PRONÓSTICO
# ------------------------------------------------------------

# Los datos de Open-Meteo son horarios.
#
# Si un vuelo sale a las 17:20, utilizaremos el
# pronóstico correspondiente a las 17:00.
#
# IMPORTANTE: el redondeo se hace después de
# convertir a UTC, no antes.

df_gold = df_gold.withColumn(
    "weather_hour_utc",
    F.date_trunc(
        "hour",
        F.col("scheduled_departure_utc")
    )
)

# ------------------------------------------------------------
# 5. CREAR VARIABLES OPERATIVAS
# ------------------------------------------------------------

# Solo utilizamos información conocida antes
# de la salida programada.

df_gold = (
    df_gold
    .withColumn(
        "is_weekend",
        F.when(
            F.col("day_of_week").isin(1, 7),
            1
        ).otherwise(0)
    )
    .withColumn(
        "is_summer",
        F.when(
            F.col("month").isin(6, 7, 8),
            1
        ).otherwise(0)
    )
    .withColumn(
        "is_peak_hour",
        F.when(
            F.col("departure_hour").between(16, 21),
            1
        ).otherwise(0)
    )
)

# ------------------------------------------------------------
# 6. REVISAR LOS RESULTADOS
# ------------------------------------------------------------

print("=" * 55)
print("PREPARACIÓN DE GOLD - VARIABLES OPERATIVAS")
print("=" * 55)

print("Número de columnas:", len(df_gold.columns))

display(
    df_gold.select(
        "flight_date",
        "origin",
        "dest",
        "airline",
        "origin_timezone",
        "scheduled_departure_local",
        "scheduled_departure_utc",
        "weather_hour_utc",
        "month",
        "departure_hour",
        "is_weekend",
        "is_summer",
        "is_peak_hour",
        "delayed_15"
    ).limit(10)
)

print("PREPARACIÓN DE GOLD COMPLETADA.")


In [0]:

# ============================================================
# CELDA 2. GUARDAR VARIABLES OPERATIVAS EN GOLD
# ============================================================

# Guardamos las variables operativas que utilizaremos
# en los modelos de predicción de retrasos.
#
# La columna dep_delay se conserva para análisis,
# pero NO será utilizada como predictor de XGBoost.

# Crear el esquema Gold si todavía no existe
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.gold_flights
""")

# Guardar la tabla en formato Delta
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold_flights.flights_features")
)

print("TABLA GOLD CREADA CORRECTAMENTE")

# Verificar que la tabla exista
display(
    spark.table("workspace.gold_flights.flights_features")
    .select(
        "flight_date",
        "origin",
        "dest",
        "airline",
        "scheduled_departure_utc",
        "weather_hour_utc",
        "is_weekend",
        "is_summer",
        "is_peak_hour",
        "delayed_15"
    )
    .limit(10)
)
